# 05 · Catálogo de dados

## 1. Objetivo e método

Os comentários gravados no Unity Catalog são o dicionário do projeto: cada notebook descreve as tabelas que cria no momento em que as cria. Este notebook apenas lê esses comentários do `information_schema` (visão do Unity Catalog que lista tabelas, colunas, tipos e comentários) e os organiza em um catálogo único.

Nada é redigido aqui. Se uma descrição estiver errada, a correção vai para o notebook que criou a tabela, e o catálogo se atualiza na próxima execução.

Roda por último, depois de todas as camadas.

## 2. Configuração

In [0]:
import os
import sys

from pyspark.sql import functions as F

# Walk up from the working directory until the folder holding `src` is found,
# so the notebook works at any depth inside notebooks/
REPO_ROOT = os.getcwd()
while not os.path.isdir(os.path.join(REPO_ROOT, "src")):
    parent = os.path.dirname(REPO_ROOT)
    if parent == REPO_ROOT:
        raise FileNotFoundError("Repository root with a src folder not found above " + os.getcwd())
    REPO_ROOT = parent
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

from src.config import CATALOG, SCHEMA_BRONZE, SCHEMA_SILVER, SCHEMA_GOLD

# Layers in reading order. Silver and Gold must describe every column; Bronze lists
# only the columns documented from the ANEEL dictionaries
SCHEMAS = [SCHEMA_BRONZE, SCHEMA_SILVER, SCHEMA_GOLD]
SCHEMAS_COM_COLUNAS = [SCHEMA_SILVER, SCHEMA_GOLD]
ORDEM_SCHEMA = {s: i for i, s in enumerate(SCHEMAS)}

# Markers delimiting the markdown block that src/gerar_catalogo.py extracts
# from the exported notebook into docs/catalogo_dados.md
INICIO_MD = "<!-- CATALOGO_INICIO -->"
FIM_MD = "<!-- CATALOGO_FIM -->"

## 3. Tabelas

Uma linha por tabela, com a descrição gravada no Unity Catalog e o volume atual. Tabelas de trabalho descartadas ao fim de cada notebook não aparecem, porque já não existem.

In [0]:
tabelas = (spark.table(f"{CATALOG}.information_schema.tables")
    .filter(F.col("table_schema").isin(SCHEMAS))
    .select(F.col("table_schema").alias("camada"),
            F.col("table_name").alias("tabela"),
            F.col("table_type").alias("tipo"),
            F.col("comment").alias("descricao"))
    .toPandas())

# Row counts come from each table; on Delta this reads metadata, not the data
tabelas["linhas"] = [
    spark.table(f"{CATALOG}.{c}.{t}").count() for c, t in zip(tabelas["camada"], tabelas["tabela"])
]
tabelas["ordem"] = tabelas["camada"].map(ORDEM_SCHEMA)
tabelas = tabelas.sort_values(["ordem", "tabela"]).drop(columns="ordem").reset_index(drop=True)

display(tabelas)

camada,tabela,tipo,descricao,linhas
bronze,_ingestion_log,MANAGED,null,14
bronze,commercial_quality,MANAGED,null,1493698
bronze,complaints,MANAGED,null,29263282
bronze,continuity_indicators,MANAGED,null,5108332
bronze,emergency_occurrences_v1,MANAGED,null,36390229
bronze,emergency_occurrences_v2,MANAGED,null,6789484
bronze,indger_commercial,MANAGED,null,267550
bronze,indger_commercial_services,MANAGED,null,23922861
bronze,pdd_investment,MANAGED,null,5484
bronze,voltage_conformity,MANAGED,null,5506240


## 4. Colunas

Tipo e descrição de cada coluna da Silver e da Gold, na ordem em que aparecem na tabela. Da Bronze entram as colunas descritas a partir dos dicionários da ANEEL.

In [0]:
colunas = (spark.table(f"{CATALOG}.information_schema.columns")
    .filter(F.col("table_schema").isin(SCHEMAS_COM_COLUNAS) |
            ((F.col("table_schema") == SCHEMA_BRONZE) & F.col("comment").isNotNull()))
    .select(F.col("table_schema").alias("camada"),
            F.col("table_name").alias("tabela"),
            F.col("ordinal_position").alias("posicao"),
            F.col("column_name").alias("coluna"),
            F.col("full_data_type").alias("tipo"),
            F.col("comment").alias("descricao"))
    .toPandas())

colunas["ordem"] = colunas["camada"].map(ORDEM_SCHEMA)
colunas = colunas.sort_values(["ordem", "tabela", "posicao"]).drop(columns="ordem").reset_index(drop=True)

display(colunas)

camada,tabela,posicao,coluna,tipo,descricao
silver,controle_anomalias_manifestacao,0,num_cnpj,string,CNPJ da distribuidora de grande porte
silver,controle_anomalias_manifestacao,1,nivel,int,Nivel de atendimento: 1 (central de atendimento) ou 2 (ouvidoria)
silver,controle_anomalias_manifestacao,2,bloco,string,"Bloco da dim_tipologia: comercial_estrito, rede_e_qualidade_outros ou tecnico"
silver,controle_anomalias_manifestacao,3,ano_mes,int,Ano e mes de competencia no formato AAAAMM
silver,controle_anomalias_manifestacao,4,qtd_recebidas,bigint,Total publicado de recebidas no bloco e mes; zero quando nada foi publicado
silver,controle_anomalias_manifestacao,5,qtd_procedentes,bigint,Total publicado de procedentes no bloco e mes; zero quando nada foi publicado
silver,controle_anomalias_manifestacao,6,qtd_improcedentes,bigint,Total publicado de improcedentes no bloco e mes; zero quando nada foi publicado
silver,controle_anomalias_manifestacao,7,med_local_recebidas,double,Mediana de recebidas nos 3 meses anteriores e 3 seguintes do mesmo bloco
silver,controle_anomalias_manifestacao,8,med_serie_recebidas,double,Mediana de recebidas nos 30 meses da serie do mesmo bloco
silver,controle_anomalias_manifestacao,9,med_seguinte_recebidas,double,Mediana de recebidas nos ate 3 meses seguintes; confirma salto persistente


## 5. Testes

O catálogo só cumpre seu papel se estiver completo. Três verificações: toda tabela da Silver e da Gold tem descrição, toda coluna delas tem descrição, e nenhuma tabela está vazia.

In [0]:
testes = []

silver_gold = tabelas[tabelas["camada"].isin(SCHEMAS_COM_COLUNAS)]

tabelas_sem = silver_gold[silver_gold["descricao"].isna() | (silver_gold["descricao"].str.strip() == "")]
testes.append(("toda tabela da Silver e da Gold descrita",
               len(tabelas_sem) == 0,
               f"{len(tabelas_sem)} tabelas sem descricao: " + ", ".join(tabelas_sem["tabela"])))

colunas_sg = colunas[colunas["camada"].isin(SCHEMAS_COM_COLUNAS)]
colunas_sem = colunas_sg[colunas_sg["descricao"].isna() | (colunas_sg["descricao"].str.strip() == "")]
lista_sem = ", ".join(f"{t}.{c}" for t, c in zip(colunas_sem["tabela"], colunas_sem["coluna"]))
testes.append(("toda coluna da Silver e da Gold descrita",
               len(colunas_sem) == 0,
               f"{len(colunas_sem)} colunas sem descricao: {lista_sem}"))

vazias = tabelas[tabelas["linhas"] == 0]
testes.append(("nenhuma tabela vazia",
               len(vazias) == 0,
               f"{len(vazias)} tabelas vazias: " + ", ".join(vazias["tabela"])))

for nome, passou, detalhe in testes:
    print(f"[{'OK' if passou else 'FALHOU':<7}] {nome:<45} {detalhe}")

if all(p for _, p, _ in testes):
    print("\nCatalogo completo.")
else:
    print("\nHa teste sem passar; completar os comentarios no notebook que cria a tabela.")

[OK     ] toda tabela da Silver e da Gold descrita      0 tabelas sem descricao: 
[OK     ] toda coluna da Silver e da Gold descrita      0 colunas sem descricao: 
[OK     ] nenhuma tabela vazia                          0 tabelas vazias: 

Catalogo completo.


## 6. Catálogo em markdown

Gera o texto que vai para `docs/catalogo_dados.md`. O script `src/gerar_catalogo.py` lê este bloco do notebook exportado; ninguém copia à mão.

In [0]:
def celula(texto):
    """Make a value safe for a markdown table cell."""
    if texto is None or (isinstance(texto, float) and texto != texto):
        return ""
    return str(texto).replace("|", "\\|").replace("\n", " ").strip()

nomes_camada = {SCHEMA_BRONZE: "Bronze", SCHEMA_SILVER: "Silver", SCHEMA_GOLD: "Gold"}

linhas_md = [
    "# Catálogo de dados",
    "",
    f"Gerado do `information_schema` do catálogo `{CATALOG}` no Unity Catalog pelo notebook "
    "`05_catalogo_dados`. As descrições são os comentários gravados por cada notebook ao criar a tabela.",
    "",
    "## Tabelas",
    "",
    "| Camada | Tabela | Linhas | Descrição |",
    "|---|---|---:|---|",
]
for _, r in tabelas.iterrows():
    qtd = f"{r['linhas']:,}".replace(",", ".")
    linhas_md.append(f"| {nomes_camada[r['camada']]} | `{r['tabela']}` | {qtd} | {celula(r['descricao'])} |")

for camada in SCHEMAS:
    bloco = colunas[colunas["camada"] == camada]
    if bloco.empty:
        continue
    linhas_md += ["", f"## {nomes_camada[camada]}"]
    for tabela, cols in bloco.groupby("tabela", sort=True):
        descricao = tabelas.loc[(tabelas["camada"] == camada) & (tabelas["tabela"] == tabela), "descricao"]
        linhas_md += ["", f"### `{camada}.{tabela}`", ""]
        if len(descricao) and celula(descricao.iloc[0]):
            linhas_md += [celula(descricao.iloc[0]), ""]
        linhas_md += ["| Coluna | Tipo | Descrição |", "|---|---|---|"]
        for _, c in cols.iterrows():
            linhas_md.append(f"| `{c['coluna']}` | `{c['tipo']}` | {celula(c['descricao'])} |")

print(INICIO_MD)
print("\n".join(linhas_md))
print(FIM_MD)

<!-- CATALOGO_INICIO -->
# Catálogo de dados

Gerado do `information_schema` do catálogo `mvp_aneel` no Unity Catalog pelo notebook `05_catalogo_dados`. As descrições são os comentários gravados por cada notebook ao criar a tabela.

## Tabelas

| Camada | Tabela | Linhas | Descrição |
|---|---|---:|---|
| Bronze | `_ingestion_log` | 14 |  |
| Bronze | `commercial_quality` | 1.493.698 |  |
| Bronze | `complaints` | 29.263.282 |  |
| Bronze | `continuity_indicators` | 5.108.332 |  |
| Bronze | `emergency_occurrences_v1` | 36.390.229 |  |
| Bronze | `emergency_occurrences_v2` | 6.789.484 |  |
| Bronze | `indger_commercial` | 267.550 |  |
| Bronze | `indger_commercial_services` | 23.922.861 |  |
| Bronze | `pdd_investment` | 5.484 |  |
| Bronze | `voltage_conformity` | 5.506.240 |  |
| Silver | `controle_anomalias_manifestacao` | 5.940 | Teste de meses anomalos das reclamacoes por distribuidora de grande porte, nivel, bloco      e mes. Evidencia da imputacao do comercial estrito no nivel 1

## Autoavaliação desta etapa

**O que esta etapa entregou.** 

O catálogo de dados do projeto, gerado a partir do `information_schema` do Unity Catalog: todas as tabelas das camadas Bronze, Silver e Gold, com descrição e volume, e, para a Silver e a Gold, cada coluna com tipo e descrição. 

Nada é redigido aqui: as descrições são os comentários gravados por cada notebook no momento em que cria a tabela, e o catálogo se atualiza a cada execução. Os testes verificam que nenhuma tabela ou coluna da Silver e da Gold ficou sem descrição e que nenhuma tabela está vazia. A descrição das parcelas vem do domínio de indicadores publicado pela ANEEL. O mesmo notebook imprime o catálogo em markdown, que o `src/gerar_catalogo.py` grava em `docs/catalogo_dados.md`.